# 2026-10-21 포커 실험 1·2·3단계 검토
원본 JSONL을 읽기만 하는 표준 라이브러리 분석이다. 모델/서버/임베딩을 호출하지 않는다.
정식 30사례 실행만 포함하며 소량 Ollama/OpenAI 연결 검사는 제외한다.
1·2단계는 한 번의 판단, 3단계는 독립 라운드이며 장기 학습 실험이 아니다.
행동 차이는 의미 있는 근거 활용이나 성과 향상의 증명이 아니다. review.csv 수동 점수는 비어 있다.
같은 사례의 input_hash는 게임/요청 ID를 제외한 판세 동일성을 검사한다. 실제 프롬프트의 ID까지 동일하다는 뜻은 아니다.
검색 사건 수는 캐시 재사용을 포함한 등장 횟수이며 서로 다른 과거 사건 수가 아니다.
시간은 클라이언트 벽시계이며 중첩 구간을 합산하지 않는다.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import json
import statistics as stats

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/results").is_dir())
RUNS = {1: "20261021T025730-f3be3013", 2: "20261021T033836-21cd2903", 3: "20261021T041257-fab9483c"}
def rows(stage, stream):
    path = ROOT / "experiments/results" / RUNS[stage] / (stream + ".jsonl")
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()] if path.exists() else []
def manifest(stage):
    return json.loads((ROOT / "experiments/results" / RUNS[stage] / "manifest.json").read_text())
def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))

overview = []
for stage in RUNS:
    trial_rows = rows(stage, "trials")
    meta = manifest(stage)
    requests = [r for r in rows(stage, "model_calls") if r["event"] == "request"]
    responses = [r for r in rows(stage, "model_calls") if r["event"] == "response"]
    assert len(trial_rows) == meta["planned_trials"]
    assert len({r["trial_id"] for r in trial_rows}) == len(trial_rows)
    assert len({r["call_id"] for r in requests}) == len(requests)
    assert len({r["call_id"] for r in responses}) == len(responses)
    assert {r["call_id"] for r in requests} == {r["call_id"] for r in responses}
    paired = defaultdict(list)
    for row in trial_rows:
        paired[row["case_id"]].append(row)
    assert all(len({r["input_hash"] for r in group}) == 1 for group in paired.values())
    assert all(len({r["bank_hash"] for r in group}) == 1 for group in paired.values())
    assert all(not any(r["saved_memos"].values()) for r in trial_rows if stage != 1 and not r["memo_input_enabled"])
    timing = [r["duration_s"] for r in rows(stage, "timings") if r["kind"] == "model"]
    ordered = sorted(timing)
    overview.append(dict(stage=stage, run=RUNS[stage], model=meta["model"], valid=sum(r["valid"] for r in trial_rows),
                         attempted=len(trial_rows), cases=len(paired), duration_s=meta["total_duration_s"],
                         model_calls=len(requests), mean_model_s=stats.mean(timing),
                         p95_model_s=ordered[__import__("math").ceil(len(ordered)*.95)-1],
                         model_wall_share=sum(timing)/meta["total_duration_s"]))
assert len({manifest(s)["model_digest"] for s in RUNS}) == 1
for file in ("agents/graph.py", "agents/poker_llm.py", "agents/runner.py", "agents/table.py"):
    assert len({manifest(s)["source_hashes"][file] for s in RUNS}) == 1
show(overview)
print("Total recorded seconds:", sum(r["duration_s"] for r in overview))


[
  {
    "stage": 1,
    "run": "20261021T025730-f3be3013",
    "model": "qwen3:8b",
    "valid": 90,
    "attempted": 90,
    "cases": 30,
    "duration_s": 1158.496366542,
    "model_calls": 90,
    "mean_model_s": 12.278943482811094,
    "p95_model_s": 17.936466582999856,
    "model_wall_share": 0.9539131458406125
  },
  {
    "stage": 2,
    "run": "20261021T033836-21cd2903",
    "model": "qwen3:8b",
    "valid": 120,
    "attempted": 120,
    "cases": 30,
    "duration_s": 1380.9028347910007,
    "model_calls": 122,
    "mean_model_s": 9.919915170434475,
    "p95_model_s": 12.698899332999645,
    "model_wall_share": 0.876404639270781
  },
  {
    "stage": 3,
    "run": "20261021T041257-fab9483c",
    "model": "qwen3:8b",
    "valid": 119,
    "attempted": 120,
    "cases": 60,
    "duration_s": 6675.846017458998,
    "model_calls": 658,
    "mean_model_s": 9.834303460186879,
    "p95_model_s": 16.951573417001782,
    "model_wall_share": 0.9693111045221483
  }
]
Total recorded sec

In [2]:
# Stage 1/2: changes in executed action and effective raise increment, not quality scores.
changes = []
for stage in (1, 2):
    grouped = defaultdict(dict)
    for row in rows(stage, "trials"):
        grouped[row["case_id"]][row["condition"]] = row
    baseline = "no_information" if stage == 1 else "control"
    conditions = ("relevant_history", "unrelated_history") if stage == 1 else ("search_only", "memo_only", "both")
    def signature(row):
        bet = row["betting_signature"]
        return bet["action"], bet["actual_increment"]
    for condition in conditions:
        pairs = [(arms[baseline], arms[condition]) for arms in grouped.values()]
        assert all(a["valid"] and b["valid"] for a, b in pairs)
        changes.append(dict(stage=stage, condition=condition, pairs=len(pairs),
                            changed=sum(signature(a) != signature(b) for a, b in pairs)))
show(changes)


[
  {
    "stage": 1,
    "condition": "relevant_history",
    "pairs": 30,
    "changed": 25
  },
  {
    "stage": 1,
    "condition": "unrelated_history",
    "pairs": 30,
    "changed": 25
  },
  {
    "stage": 2,
    "condition": "search_only",
    "pairs": 30,
    "changed": 22
  },
  {
    "stage": 2,
    "condition": "memo_only",
    "pairs": 30,
    "changed": 15
  },
  {
    "stage": 2,
    "condition": "both",
    "pairs": 30,
    "changed": 17
  }
]


In [3]:
# Stage 3: one independent round per condition/case, paired within opponent.
performance, paired_results = [], []
trial_rows, events = rows(3, "trials"), rows(3, "events")
for opponent in ("B", "C"):
    for condition in ("control", "both"):
        group = [r for r in trial_rows if r["opponent"] == opponent and r["condition"] == condition]
        ids = {r["trial_id"] for r in group}
        actions = Counter(r["event"]["action"] for r in events if r["trial_id"] in ids
                          and r["event"]["type"] == "BetPlaced" and r["event"]["player_id"] == "A")
        performance.append(dict(opponent=opponent, condition=condition, n=len(group),
                                valid=sum(r["valid"] for r in group), wins=sum(r["winner"] == "A" for r in group),
                                mean_chip_delta=stats.mean(r["chip_delta"] for r in group), actions=dict(actions)))
    arms = defaultdict(dict)
    for row in trial_rows:
        if row["opponent"] == opponent:
            arms[row["case_id"]][row["condition"]] = row
    pairs = list(arms.values())
    diffs = [a["both"]["chip_delta"] - a["control"]["chip_delta"] for a in pairs]
    valid_diffs = [a["both"]["chip_delta"] - a["control"]["chip_delta"] for a in pairs if all(r["valid"] for r in a.values())]
    paired_results.append(dict(opponent=opponent, pairs=len(pairs), valid_pairs=len(valid_diffs),
                               winners_changed=sum(a["both"]["winner"] != a["control"]["winner"] for a in pairs),
                               mean_difference_with_fallback=stats.mean(diffs), mean_difference_valid=stats.mean(valid_diffs),
                               differences=dict(Counter(diffs))))
show(performance)
show(paired_results)


[
  {
    "opponent": "B",
    "condition": "control",
    "n": 30,
    "valid": 29,
    "wins": 19,
    "mean_chip_delta": 1.1333333333333333,
    "actions": {
      "raise": 37,
      "check": 1,
      "fold": 1
    }
  },
  {
    "opponent": "B",
    "condition": "both",
    "n": 30,
    "valid": 30,
    "wins": 19,
    "mean_chip_delta": 1,
    "actions": {
      "raise": 38
    }
  },
  {
    "opponent": "C",
    "condition": "control",
    "n": 30,
    "valid": 30,
    "wins": 22,
    "mean_chip_delta": 0.5666666666666667,
    "actions": {
      "raise": 30
    }
  },
  {
    "opponent": "C",
    "condition": "both",
    "n": 30,
    "valid": 30,
    "wins": 22,
    "mean_chip_delta": 0.43333333333333335,
    "actions": {
      "raise": 30
    }
  }
]
[
  {
    "opponent": "B",
    "pairs": 30,
    "valid_pairs": 29,
    "winners_changed": 0,
    "mean_difference_with_fallback": -0.13333333333333333,
    "mean_difference_valid": 0,
    "differences": {
      "0": 27,
      "-4": 

In [4]:
# Data-path audit: retrieval content, memo persistence, manual review, failures.
quality = []
for stage in RUNS:
    research = rows(stage, "research")
    matches = [match for r in research for match in r["response"].get("matches", [])]
    tools = rows(stage, "tools")
    writes = [r for r in tools if r["name"] == "write_memo" and r["result"].get("written")]
    readbacks = [r for r in tools if r["name"] == "memo_read_back"]
    assert len(writes) == len(readbacks)
    expected_writes = Counter((r["trial_id"], r.get("decision_id"), r["arguments"]["scope"], r["arguments"]["text"]) for r in writes)
    actual_writes = Counter((r["trial_id"], r.get("decision_id"), r["result"]["scope"], r["result"]["text"]) for r in readbacks)
    assert expected_writes == actual_writes
    review_path = ROOT / "experiments/results" / RUNS[stage] / "review.csv"
    with review_path.open(newline="") as handle:
        review = list(csv.DictReader(handle))
    quality.append(dict(stage=stage, research_status=dict(Counter(r["response"]["status"] for r in research)),
                        retrieved_match_occurrences=len(matches),
                        retrieved_event_types=dict(Counter(m["metadata"]["event_type"] for m in matches)),
                        successful_writes=len(writes), verified_readbacks=len(readbacks),
                        manually_scored=sum(bool(r.get("speech_score") or r.get("memo_score")) for r in review)))
    trial_map = {r["trial_id"]: r for r in rows(stage, "trials")}
    for request in rows(stage, "model_calls"):
        if request["event"] == "request":
            matches = json.loads(request["user"]).get("past_experiences", {}).get("matches", [])
            assert not matches or trial_map[request["trial_id"]]["search_enabled"]
show(quality)
show([{k: r.get(k) for k in ("trial_id", "runner_errors", "chip_delta", "winner")} for r in rows(3, "trials") if not r["valid"]])
for stage, case_id, conditions in [(2, "s2-BLUFF-01", ("control", "both")), (2, "s2-CONSERVATIVE-01", ("control", "memo_only"))]:
    for row in rows(stage, "trials"):
        if row["case_id"] == case_id and row["condition"] in conditions:
            show(dict(trial_id=row["trial_id"], bet=row["betting_signature"], speech=row["decision"]["text"], saved_memos=row["saved_memos"]))


[
  {
    "stage": 1,
    "research_status": {
      "ready": 60,
      "skipped": 30
    },
    "retrieved_match_occurrences": 156,
    "retrieved_event_types": {
      "fixture_public_bundle": 156
    },
    "successful_writes": 270,
    "verified_readbacks": 270,
    "manually_scored": 0
  },
  {
    "stage": 2,
    "research_status": {
      "skipped": 60,
      "ready": 60
    },
    "retrieved_match_occurrences": 180,
    "retrieved_event_types": {
      "Speech": 138,
      "BetPlaced": 42
    },
    "successful_writes": 180,
    "verified_readbacks": 180,
    "manually_scored": 0
  },
  {
    "stage": 3,
    "research_status": {
      "skipped": 321,
      "ready": 198
    },
    "retrieved_match_occurrences": 594,
    "retrieved_event_types": {
      "Speech": 531,
      "BetPlaced": 63
    },
    "successful_writes": 919,
    "verified_readbacks": 919,
    "manually_scored": 0
  }
]
[
  {
    "trial_id": "s3-B-07-control",
    "runner_errors": {
      "A": "ValueError: invali

In [5]:
# Count each HTTP request/response once by call_id; include retries and post-round reflection.
token_rows = []
for folder in sorted((ROOT / "experiments/results").iterdir()):
    if not (folder / "manifest.json").exists():
        continue
    meta = json.loads((folder / "manifest.json").read_text())
    log = folder / "model_calls.jsonl"
    calls = [json.loads(line) for line in log.read_text().splitlines() if line.strip()] if log.exists() else []
    requests = [r for r in calls if r["event"] == "request"]
    responses = [r for r in calls if r["event"] == "response"]
    assert len(requests) == len({r["call_id"] for r in requests})
    assert len(responses) == len({r["call_id"] for r in responses})
    assert {r["call_id"] for r in requests} == {r["call_id"] for r in responses}
    assert all(isinstance(r.get("usage", {}).get("tokens"), int) for r in responses)
    token_rows.append(dict(run=folder.name, formal=folder.name in RUNS.values(), stage=meta["config"]["stage"],
                           model=meta.get("model"), requests=len(requests),
                           input_plus_output_tokens=sum(r["usage"]["tokens"] for r in responses)))
show(token_rows)
for label, selected in (("formal", [r for r in token_rows if r["formal"]]), ("all_logged", token_rows)):
    show(dict(scope=label, requests=sum(r["requests"] for r in selected),
              input_plus_output_tokens=sum(r["input_plus_output_tokens"] for r in selected)))
print("Only recorded experiment calls are counted. Embedding usage and unlogged standalone demo calls are not recoverable here.")
print("Ollama logs retain combined input/output usage, not separate token counts.")


[
  {
    "run": "20261021T024041-0d92de4e",
    "formal": false,
    "stage": 1,
    "model": "qwen3:8b",
    "requests": 3,
    "input_plus_output_tokens": 8416
  },
  {
    "run": "20261021T024221-2f96eb3a",
    "formal": false,
    "stage": 2,
    "model": "qwen3:8b",
    "requests": 1,
    "input_plus_output_tokens": 2345
  },
  {
    "run": "20261021T024351-74a61922",
    "formal": false,
    "stage": 3,
    "model": "qwen3:8b",
    "requests": 19,
    "input_plus_output_tokens": 29966
  },
  {
    "run": "20261021T025730-f3be3013",
    "formal": true,
    "stage": 1,
    "model": "qwen3:8b",
    "requests": 90,
    "input_plus_output_tokens": 243490
  },
  {
    "run": "20261021T033836-21cd2903",
    "formal": true,
    "stage": 2,
    "model": "qwen3:8b",
    "requests": 122,
    "input_plus_output_tokens": 247106
  },
  {
    "run": "20261021T040020-d1ab91ee",
    "formal": false,
    "stage": 1,
    "model": "gpt-6-luna",
    "requests": 1,
    "input_plus_output_tokens": 236

In [6]:
# Read existing Chroma SQLite metadata without opening the Chroma client or calling embeddings.
import sqlite3
retrieval_audit = []
for stage in (2, 3):
    db_counts = []
    for trial in rows(stage, "trials"):
        if not trial["search_enabled"]:
            continue
        stored_path = Path(trial["game_config"]["data_dir"]) / "chroma/chroma.sqlite3"
        db = ROOT / "experiments/results" / RUNS[stage] / "data" / trial["trial_id"] / "chroma/chroma.sqlite3"
        if not db.exists():
            db = stored_path
        assert db.exists(), trial["trial_id"]
        with sqlite3.connect(db.resolve().as_uri() + "?mode=ro", uri=True) as conn:
            db_counts.append(conn.execute("select count(*) from embedding_metadata where key='event_type' and string_value='RoundEnded'").fetchone()[0])
    searches = [r for r in rows(stage, "tools") if r["name"] == "search_experiences"]
    identical_claims = 0
    types = Counter()
    for search in searches:
        events = [json.loads(m["text"].split("\n", 1)[1]) for m in search["result"]["matches"]]
        types.update(e["type"] for e in events)
        texts = [event.get("text") for event in events]
        identical_claims += bool(texts) and all(texts) and len(set(texts)) == 1
    retrieval_audit.append(dict(stage=stage, databases_checked=len(db_counts),
                                indexed_roundended_min=min(db_counts), indexed_roundended_max=max(db_counts),
                                actual_search_calls=len(searches), returned_event_types=dict(types),
                                all_hits_same_speech=identical_claims))
show(retrieval_audit)


[
  {
    "stage": 2,
    "databases_checked": 60,
    "indexed_roundended_min": 20,
    "indexed_roundended_max": 20,
    "actual_search_calls": 60,
    "returned_event_types": {
      "Speech": 138,
      "BetPlaced": 42
    },
    "all_hits_same_speech": 42
  },
  {
    "stage": 3,
    "databases_checked": 60,
    "indexed_roundended_min": 20,
    "indexed_roundended_max": 20,
    "actual_search_calls": 88,
    "returned_event_types": {
      "Speech": 243,
      "BetPlaced": 21
    },
    "all_hits_same_speech": 81
  }
]
